# 🌿 FitoBot — Passifloraceae
### Bot de consulta sobre enfermedades y fungicidas con fine-tuning

## 1️⃣ Instalación de dependencias

In [1]:
!pip install pandas gradio sentence-transformers faiss-cpu transformers \
    torch accelerate bitsandbytes peft trl datasets huggingface_hub -q
print("✅ Dependencias instaladas")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 751.0/751.0 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 21.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.5 MB/s eta 0:00:00
✅ Dependencias instaladas


## 2️⃣ Configuración del token y Drive

In [2]:
from google.colab import userdata, drive
import os

# Montar Drive
drive.mount('/content/drive')

# Token Hugging Face
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
print("✅ Token configurado")

Mounted at /content/drive
✅ Token configurado


## 3️⃣ Cargar y limpiar el dataset CSV

In [12]:
import pandas as pd
import re

df = pd.read_csv('/content/drive/MyDrive/Proyecto_MIE/fungicidas.csv', encoding='utf-8')
print(f"✅ Cargadas {len(df)} filas")
print(df[['Especie','Enfermedad','Fungicida']].to_string())

def limpiar(texto):
    texto = str(texto)
    texto = re.sub(r'\$.*?\$', '', texto)   # quitar fórmulas LaTeX
    texto = re.sub(r'\\[a-z]+', '', texto)  # quitar comandos LaTeX
    texto = re.sub(r'\s+', ' ', texto).strip()
    return texto

# Aplicar limpieza a todo el dataframe
for col in df.columns:
    df[col] = df[col].apply(limpiar)

print("\n✅ Datos limpios")

EmptyDataError: No columns to parse from file

In [17]:
import os

# List contents of the directory to check for the file
drive_path = '/content/drive/MyDrive/Proyecto MIE/primer/fungicidas.csv' # Changed to the directory path
if os.path.exists(drive_path):
    print(f"Contenido de '{drive_path}':")
    for item in os.listdir(drive_path):
        print(item)
else:
    print(f"El directorio '{drive_path}' no existe. Por favor, verifica la ruta.")

Contenido de '/content/drive/MyDrive/Proyecto MIE/primer/fungicidas.csv':


NotADirectoryError: [Errno 20] Not a directory: '/content/drive/MyDrive/Proyecto MIE/primer/fungicidas.csv'

## 4️⃣ Generar dataset de fine-tuning (pares pregunta-respuesta)

In [4]:
import json

def especie_corta(texto):
    if 'Maracuyá' in texto or 'flavicarpa' in texto: return 'maracuyá'
    if 'Gulupa' in texto: return 'gulupa'
    if 'Granadilla' in texto or 'ligularis' in texto: return 'granadilla'
    return 'Passiflora spp.'

pares = []

for _, row in df.iterrows():
    ec = especie_corta(row['Especie'])
    enf_full = row['Enfermedad']
    enf = enf_full.split('(')[0].strip()
    pat_m = re.search(r'\((.*?)\)', enf_full)
    pat = pat_m.group(1) if pat_m else ''
    fun = row['Fungicida']
    met = row['Metodo de aplicacion']
    mod = row['Tipo y Modo de accion']
    rie = row['Riesgos en la salud']
    pre = row['Como evitar esos riesgos']

    preguntas_respuestas = [
        # Enfermedad → tratamiento
        (f"¿Qué fungicida uso para la {enf} en {ec}?",
         f"Para controlar la {enf} en {ec} se recomienda {fun}, aplicado por {met}. Recuerda: {pre}"),
        (f"Tengo {enf} en mi cultivo de {ec}, ¿qué hago?",
         f"Usa {fun} mediante {met}. Ten en cuenta estos riesgos: {rie}. Para protegerte: {pre}"),
        (f"¿Cómo trato la {enf} en {ec}?",
         f"El control de {enf} en {ec} se hace con {fun} ({met}). Modo de acción: {mod}."),
        # Agente causal
        (f"¿Qué causa la {enf} en {ec}?" if not pat else f"¿Qué causa la {enf}?",
         f"La {enf} es causada por {pat}. Se controla con {fun} aplicado por {met}." if pat
         else f"La {enf} afecta a {ec} y se controla con {fun} aplicado por {met}."),
        # Riesgos del fungicida
        (f"¿Qué riesgos tiene el {fun}?",
         f"El {fun} presenta: {rie}. Para minimizar el riesgo: {pre}"),
        (f"¿Es peligroso el {fun} para la salud?",
         f"Sobre el {fun}: {rie}. Medidas de protección: {pre}"),
        (f"¿Cómo me protejo al aplicar {fun}?",
         f"Al usar {fun} debes saber: {rie}. Protección recomendada: {pre}"),
        (f"¿El {fun} daña la piel o los ojos?",
         f"Riesgos del {fun}: {rie}. Se recomienda: {pre}"),
        # Aplicación
        (f"¿Cómo se aplica el {fun}?",
         f"El {fun} se aplica mediante {met}. Acción: {mod}. Precauciones: {pre}"),
        (f"¿Cuál es el modo de acción del {fun}?",
         f"El {fun} actúa como: {mod}. Controla {enf} en {ec}."),
        # EPP y seguridad
        (f"¿Qué equipo de protección necesito para aplicar {fun}?",
         f"Para aplicar {fun} ten en cuenta: {rie}. Equipo necesario: {pre}"),
        # General
        (f"¿Qué enfermedades trata el {fun}?",
         f"El {fun} controla {enf} en {ec}, aplicado por {met}."),
    ]

    for pregunta, respuesta in preguntas_respuestas:
        # Formato Mistral instruct
        texto = f"<s>[INST] {pregunta} [/INST] {respuesta} </s>"
        pares.append({"text": texto, "pregunta": pregunta, "respuesta": respuesta})

print(f"✅ Dataset generado: {len(pares)} pares pregunta-respuesta")

# Guardar JSONL
ruta_jsonl = '/content/drive/MyDrive/Proyecto_MIE/fitobot_dataset.jsonl'
with open(ruta_jsonl, 'w', encoding='utf-8') as f:
    for p in pares:
        f.write(json.dumps({"text": p["text"]}, ensure_ascii=False) + '\n')

# Guardar CSV para revisión
ruta_csv = '/content/drive/MyDrive/Proyecto_MIE/fitobot_dataset_revision.csv'
pd.DataFrame(pares).to_csv(ruta_csv, index=False)

print(f"✅ Guardado en Drive: {ruta_jsonl}")
print("\n=== MUESTRA ===")
for p in pares[:3]:
    print(f"P: {p['pregunta']}")
    print(f"R: {p['respuesta']}")
    print()

NameError: name 'df' is not defined

## 5️⃣ Fine-tuning con QLoRA (eficiente en Colab GPU T4)

In [2]:
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
import torch

# ─── Modelo base ───────────────────────────────────────────────
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
OUTPUT_DIR = "/content/drive/MyDrive/Proyecto_MIE/fitobot-finetuned"

# ─── Cuantización 4-bit para caber en Colab T4 ─────────────────
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("⏳ Cargando tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=os.environ["HF_TOKEN"])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("⏳ Cargando modelo en 4-bit (puede tardar 3-5 min)...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)
model = prepare_model_for_kbit_training(model)

# ─── Configuración LoRA ────────────────────────────────────────
lora_config = LoraConfig(
    r=16,                      # rango: mayor = más capacidad, más VRAM
    lora_alpha=32,
    target_modules=["q_proj", "v_proj", "k_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

# ─── Dataset ──────────────────────────────────────────────────
dataset = load_dataset(
    "json",
    data_files='/content/drive/MyDrive/Proyecto_MIE/fitobot_dataset.jsonl',
    split="train"
)
# Split 90% train / 10% validación
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"✅ Train: {len(dataset['train'])} | Val: {len(dataset['test'])}")

# ─── Argumentos de entrenamiento ──────────────────────────────
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,            # con 198 pares, 5 épocas es suficiente
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, # batch efectivo = 16
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    fp16=True,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    report_to="none",
    optim="paged_adamw_8bit",      # optimizador eficiente en memoria
)

# ─── Trainer ──────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=512,
    packing=False,
)

print("🚀 Iniciando fine-tuning...")
trainer.train()

# ─── Guardar modelo ───────────────────────────────────────────
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"✅ Modelo guardado en: {OUTPUT_DIR}")

ModuleNotFoundError: No module named 'trl'

## 6️⃣ Embeddings FAISS sobre el CSV (RAG)

In [1]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

def construir_ficha(row):
    return (
        f"Cultivo: {row['Especie']}\n"
        f"Enfermedad: {row['Enfermedad']}\n"
        f"Fungicida: {row['Fungicida']}\n"
        f"Aplicación: {row['Metodo de aplicacion']}\n"
        f"Riesgos: {row['Riesgos en la salud']}\n"
        f"Protección: {row['Como evitar esos riesgos']}"
    )

fichas = df.apply(construir_ficha, axis=1).tolist()

print("⏳ Generando embeddings...")
corpus_emb = embedding_model.encode(fichas, show_progress_bar=True).astype('float32')

index_faiss = faiss.IndexFlatL2(corpus_emb.shape[1])
index_faiss.add(corpus_emb)
print(f"✅ Índice FAISS listo con {index_faiss.ntotal} fichas")

ModuleNotFoundError: No module named 'faiss'

## 7️⃣ Cargar modelo fine-tuneado e inferencia RAG

In [ ]:
from peft import PeftModel

print("⏳ Cargando modelo fine-tuneado...")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)
ft_model = PeftModel.from_pretrained(base_model, OUTPUT_DIR)
ft_tokenizer = AutoTokenizer.from_pretrained(OUTPUT_DIR)
ft_tokenizer.pad_token = ft_tokenizer.eos_token
print("✅ Modelo fine-tuneado cargado")

def buscar_contexto(pregunta, k=3, umbral=2.0):
    q_emb = embedding_model.encode([pregunta]).astype('float32')
    dist, idx = index_faiss.search(q_emb, k)
    resultado = []
    for i, id_ in enumerate(idx[0]):
        if dist[0][i] < umbral:
            resultado.append(fichas[id_])
    return "\n\n".join(resultado) if resultado else None

def responder(pregunta, history):
    contexto = buscar_contexto(pregunta)

    if not contexto:
        return ("No encontré información sobre eso en mi base de datos. "
                "Puedes preguntar sobre antracnosis, moho gris, secadera, roña, "
                "mildiu, o fungicidas como Mancozeb, Azoxistrobina, Tebuconazol.")

    prompt = (
        f"<s>[INST] Eres FitoBot, asistente agrónomo experto en enfermedades "
        f"de maracuyá, gulupa y granadilla. Responde en español claro y práctico, "
        f"como si hablaras con un agricultor. Usa solo la información del contexto.\n\n"
        f"Contexto:\n{contexto}\n\n"
        f"Pregunta: {pregunta} [/INST]"
    )

    inputs = ft_tokenizer(prompt, return_tensors="pt").to(ft_model.device)
    with torch.no_grad():
        output = ft_model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.4,
            top_p=0.9,
            repetition_penalty=1.15,
            do_sample=True,
            pad_token_id=ft_tokenizer.eos_token_id
        )

    resp = ft_tokenizer.decode(output[0], skip_special_tokens=True)
    if "[/INST]" in resp:
        resp = resp.split("[/INST]", 1)[1].strip()
    return resp

print("✅ Función de respuesta lista")

## 8️⃣ Interfaz Gradio

In [ ]:
import gradio as gr

ejemplos = [
    "¿Qué fungicida uso para la antracnosis en maracuyá?",
    "¿Qué riesgos tiene el Mancozeb para la salud?",
    "¿Cómo me protejo al aplicar Tebuconazol?",
    "Tengo moho gris en gulupa, ¿qué hago?",
    "¿El Tebuconazol puede afectar el embarazo?",
    "¿Cómo controlo la secadera en granadilla?",
    "¿Qué EPP necesito para aplicar Prochloraz?",
]

chatbot_ia = gr.ChatInterface(
    fn=responder,
    chatbot=gr.Chatbot(type='messages', height=500),
    title="🌿 FitoBot — Enfermedades y Fungicidas en Passifloraceae",
    description=(
        "Consulta sobre enfermedades, fungicidas y riesgos en maracuyá, gulupa y granadilla.\n"
        "Ejemplos: '¿Qué riesgos tiene el Mancozeb?' | '¿Cómo trato la antracnosis?'"
    ),
    examples=ejemplos,
    theme="soft"
)

chatbot_ia.launch(share=True)